In [3]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated,List
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage,HumanMessage, AIMessage,SystemMessage
from dotenv import load_dotenv
import re
import json
import requests
import statistics

load_dotenv()

True

In [45]:
def liquidity_risk(ticker: str) -> dict:
    info= yf.Ticker(ticker).info

    avg_daily_volume=  info.get("averageVolume")
    float_shares=      info.get("floatShares")
    current_ratio=     info.get("currentRatio")

    return {"avg_daily_volume":  avg_daily_volume,
            "float_shares":      float_shares,
            "current_ratio":    current_ratio,}

In [46]:
def Business_risk(ticker:str)-> dict:
    symbol= yf.Ticker(ticker)
    info = symbol.info 
    financials = symbol.financials
    
    sector=info.get("sector")
    
    try:
        rev_0 = financials.loc['Total Revenue'].iloc[0]
        rev_1 = financials.loc['Total Revenue'].iloc[1]
        
        growth="Growing" if rev_0 > rev_1 else "Shrinking"
        return {
            "sector":sector,
            "revenue_growth":growth,
        }
    
    except Exception as e:
        return {"sector": sector, "revenue_growth": "Unknown", "error": str(e)}

In [ ]:
def Financial_risk (ticker:str)-> dict:
    symbol= yf.Ticker(ticker)
    info = symbol.info 
    financials = symbol.financials
    cashflow = symbol.cashflow
    balance = symbol.balance_sheet
    
    debt_to_equity= info.get("debtToEquity")
    try:
        ebit=financials.loc["EBIT"].iloc[0]  
        interest_expense=financials.loc["Interest Expense"].iloc[0]
        coverage_ratio = ebit / interest_expense
        if coverage_ratio == 0 :
            coverage_ratio = "Does not have coverage ratio"

        ocf   = cashflow.loc["Operating Cash Flow"].iloc[0]
        capex = cashflow.loc["Capital Expenditure"].iloc[0]

        fcf = ocf - capex
        
        # The Brutal Diagnosis
        if fcf < 0:
            if ocf > 0:
                diagnosis = "Negative FCF (Hyper-Growth Burn): Core ops make money, but heavy CapEx is draining cash."
            else:
                diagnosis = "Negative FCF (Lethal Burn): Core ops are bleeding cash. Danger zone."
        else:
            diagnosis = "Positive FCF: Generating surplus cash."

        current_assets = balance.loc[ 'Current Assets'].iloc[0]
        current_liabilities = balance.loc[ 'Current Liabilities'].iloc[0]
        cash= balance.loc[ 'Cash And Cash Equivalents'].iloc[0]
        current_debt=balance.loc[ 'Total Debt'].iloc[0]

        operating_nwc = (current_assets - cash) - (current_liabilities - current_debt)

        operating_nwc_status="Healthy" if operating_nwc > 0 else "Warning (Negative)"
        
        return{
            "debt_to_equity" : debt_to_equity,
            "interest_coverage": round(float(coverage_ratio),2),
            "fcf_negative":      diagnosis,
            "Operating_Net_Working_Capital":round(float(operating_nwc),2),
            "Operating_Net_Working_Capital_status":operating_nwc_status,
            "Total Debt":round(float(current_debt),2),      
        }
    except Exception as e:
        return {"debt_to_equity": debt_to_equity, "error": str(e) }




In [55]:
def market_risk(ticker: str) -> dict:
    symbol= yf.Ticker(ticker)
    info = symbol.info
    df_hist=symbol.history(period="1y")

    beta=info.get("beta")


    returns = df_hist["Close"].pct_change().dropna()

    # sort returns, take the worst (1 - confidence) percentile
    var_95 = np.percentile(returns, (1 - 0.95) * 100)
    var_99 = np.percentile(returns, (1 - 0.99) * 100)
    
    close  = df_hist["Close"]
    peak   = close.cummax()      
    trough = (close - peak) / peak 
    max_drawdown=trough.min()

    beta_signal = (
        "very high"  if beta > 1.5  else
        "high"       if beta > 1.0  else
        "moderate"   if beta > 0.5  else
        "low"
    )
    drawdown_signal = (
        "severe"   if max_drawdown < -0.30 else
        "moderate" if max_drawdown < -0.15 else
        "low"
    )

    return{
        "beta":             beta,
        "beta_signal":      beta_signal,
        "var_95":           round(float(var_95), 4),       
        "var_99":           round(float(var_99), 4),        
        "max_drawdown":     round(float(max_drawdown), 4),  
        "drawdown_signal":  drawdown_signal,  
    }

In [50]:
def run_risk_pipeline(ticker: str) -> dict:
    return {
        "ticker": ticker,
        **market_risk(ticker),
        **Financial_risk(ticker),
        **Business_risk(ticker),
        **liquidity_risk(ticker),
    }

In [61]:
run_risk_pipeline("NVDA")

{'ticker': 'NVDA',
 'beta': 2.202,
 'beta_signal': 'very high',
 'var_95': -0.037,
 'var_99': -0.0476,
 'max_drawdown': -0.2021,
 'drawdown_signal': 'moderate',
 'debt_to_equity': None,
 'interest_coverage': 547.14,
 'fcf_negative': 'Positive FCF: Generating surplus cash.',
 'Operating_Net_Working_Capital': 83836000000.0,
 'Operating_Net_Working_Capital_status': 'Healthy',
 'sector': 'Technology',
 'revenue_growth': 'Growing',
 'avg_daily_volume': 160883770,
 'float_shares': 23225224000,
 'current_ratio': 3.441}